In [2]:
import sys
from pathlib import Path

# Permite importar lista4 desde la carpeta src
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from lista4.data import load_csv
from lista4.models import linear_summary

df = load_csv("experimento_3")
df["z2"] = df["x2"] / 100

df[["x1", "x2", "z2", "y"]].describe()

,x1,x2,z2,y
count,500.000000,500.000000,500.000000,500.000000
mean,0.031513,1.062102,0.010621,0.110089
std,1.047887,101.471545,1.014715,7.405441
min,-3.747414,-289.105312,-2.891053,-20.837760
25%,-0.691469,-63.741405,-0.637414,-4.883034
50%,0.063250,0.127024,0.001270,0.125914
75%,0.720126,70.643720,0.706437,5.288287
max,3.150908,323.623238,3.236232,21.269202


In [3]:
#A
m1 = linear_summary(df, ["x1", "x2"], label="Modelo 1 (x1, x2)")
m2 = linear_summary(df, ["x1", "z2"], label="Modelo 2 (x1, z2)")

tabla_a = pd.DataFrame({
    "intercepto": [m1["intercepto"], m2["intercepto"]],
    "coef_x1": [m1["coef_x1"], m2["coef_x1"]],
    "coef_x2 / coef_z2": [m1["coef_x2"], m2["coef_z2"]],
    "riesgo_empirico": [m1["riesgo_empirico"], m2["riesgo_empirico"]],
}, index=["Modelo 1 (β)", "Modelo 2 (γ)"])

tabla_a

,intercepto,coef_x1,coef_x2 / coef_z2,riesgo_empirico
Modelo 1 (β),-0.10147,5.012066,0.050477,0.923225
Modelo 2 (γ),-0.10147,5.012066,5.047693,0.923225


In [4]:
#B
d_i = np.abs(m1["_pred"] - m2["_pred"])
max_d = d_i.max()

print(f"max_i d_i = {max_d:.3e}")

max_i d_i = 7.161e-15


### (b)

$$\max_i d_i \approx 7.2 \times 10^{-15}$$

La diferencia máxima entre las predicciones de los dos modelos es casi 0. Ambos modelos predicen exactamente lo mismo para cada observación.

In [5]:
#C
comparaciones = pd.DataFrame({
    "valor_modelo_1": [m1["intercepto"], m1["coef_x1"], 100 * m1["coef_x2"]],
    "valor_modelo_2": [m2["intercepto"], m2["coef_x1"], m2["coef_z2"]],
}, index=["γ0 vs β0", "γ1 vs β1", "γ2 vs 100·β2"])

comparaciones["error_abs"] = (comparaciones["valor_modelo_1"] - comparaciones["valor_modelo_2"]).abs()

# Los errores son tan pequeños que se imprimen aparte en notación científica
for nombre, e in comparaciones["error_abs"].items():
    print(f"Error absoluto en {nombre}: {e:.3e}")

comparaciones

Error absoluto en γ0 vs β0: 2.776e-17
Error absoluto en γ1 vs β1: 8.882e-16
Error absoluto en γ2 vs 100·β2: 1.776e-15


,valor_modelo_1,valor_modelo_2,error_abs
γ0 vs β0,-0.101470,-0.101470,2.775558e-17
γ1 vs β1,5.012066,5.012066,8.881784e-16
γ2 vs 100·β2,5.047693,5.047693,1.776357e-15


### (c) 

| Comparación | Valor modelo 1 | Valor modelo 2 | Error absoluto |
|---|---|---|---|
| γ̂₀ vs β̂₀ | -0.101470 | -0.101470 | 2.776 × 10⁻¹⁷ |
| γ̂₁ vs β̂₁ | 5.012066 | 5.012066 | 8.882 × 10⁻¹⁶ |
| γ̂₂ vs 100·β̂₂ | 5.047693 | 5.047693 | 1.776 × 10⁻¹⁵ |


In [6]:
#D
beta2 = abs(m1["coef_x2"])
gamma2 = abs(m2["coef_z2"])

print(f"|beta2|  = {beta2:.6f}")
print(f"|gamma2| = {gamma2:.6f}")
print(f"cociente |gamma2| / |beta2| = {gamma2 / beta2:.4f}")
print(f"max_i d_i = {max_d:.3e}")

|beta2|  = 0.050477
|gamma2| = 5.047693
cociente |gamma2| / |beta2| = 100.0000
max_i d_i = 7.161e-15


### (d) Comparación de coeficientes y de predicciones

| Cantidad | Valor |
|---|---|
| \|β̂₂\| | 0.050477 |
| \|γ̂₂\| | 5.047693 |
| \|γ̂₂\| / \|β̂₂\| | 100.0 |
| max dᵢ | 7.161 × 10⁻¹⁵ |

- **Coeficientes:** |γ̂₂| es 100 veces más grande que |β̂₂| (5.0477 frente a 0.0505). Esto pasa porque z2 = x2/100 es una variable 100 veces más pequeña, entonces su coeficiente tiene que ser 100 veces más grande para que el efecto sea el mismo. Por eso no se pueden comparar los coeficientes cuando las variables están en escalas distintas: que β̂₂ sea pequeño no quiere decir que x2 importe poco, solo que x2 tiene valores grandes.
- **Predicciones:** la diferencia máxima entre las predicciones de los dos modelos es del orden de 10⁻¹⁵, o sea que es cero. Entonces los dos modelos predicen lo mismo y tienen el mismo riesgo empírico. Cambiar la escala de una variable cambia su coeficiente, pero no cambia el modelo ni sus predicciones.

In [7]:
#E
tabla_e = pd.DataFrame({
    "intercepto": [m1["intercepto"], m2["intercepto"]],
    "coef_x1": [m1["coef_x1"], m2["coef_x1"]],
    "coef_x2 / coef_z2": [m1["coef_x2"], m2["coef_z2"]],
    "riesgo_empirico": [m1["riesgo_empirico"], m2["riesgo_empirico"]],
}, index=["Modelo 1 (x1, x2)", "Modelo 2 (x1, z2)"])

tabla_e["max_i d_i"] = f"{max_d:.3e}"
tabla_e

,intercepto,coef_x1,coef_x2 / coef_z2,riesgo_empirico,max_i d_i
"Modelo 1 (x1, x2)",-0.10147,5.012066,0.050477,0.923225,7.161e-15
"Modelo 2 (x1, z2)",-0.10147,5.012066,5.047693,0.923225,7.161e-15


### (e) 

| Modelo | Intercepto | Coef. de x1 | Coef. de x2 / z2 | Riesgo empírico | max dᵢ |
|---|---|---|---|---|---|
| Modelo 1 (x1, x2) | -0.101470 | 5.012066 | β̂₂ = 0.050477 | 0.923225 | 7.161 × 10⁻¹⁵ |
| Modelo 2 (x1, z2) | -0.101470 | 5.012066 | γ̂₂ = 5.047693 | 0.923225 | 7.161 × 10⁻¹⁵ |

En la tabla se ve que el intercepto, el coeficiente de x1 y el riesgo empírico son iguales en los dos modelos. Lo único que cambia es el coeficiente de la variable reescalada (γ̂₂ = 100·β̂₂), y las predicciones siguen siendo las mismas (max dᵢ ≈ 10⁻¹⁵). Es decir, dividir x2 entre 100 no cambia el modelo, solo la forma de escribirlo.